# SPM (Statistical Parametric Mapping) for Joint Kinematics

This notebook applies **Statistical Parametric Mapping (SPM1D)** to compare joint kinematic curves across groups and time points. Unlike traditional analysis that reduces a gait cycle to scalar features (peak angle, ROM), SPM tests the entire time-series curve point-by-point with proper multiple-comparison correction, identifying *where in the gait cycle* the groups differ.

**Author**: Yeon-Joo Kang | Georgia State University | 2021–2024
**Status**: Archived snapshot from dissertation research. This methodology will be revisited and developed further in a separate repository.

## Theoretical reference

The SPM approach for 1D biomechanical waveforms is described in:

> Pataky, T. C., Robinson, M. A., & Vanrenterghem, J. (2013). Vector field statistical analysis of kinematic and force trajectories. *Journal of Biomechanics*, 46(14), 2394–2401.

Implementation uses the `spm1d` Python package (Pataky lab).

## Input dependencies

This notebook expects the per-subject ensemble-mean CSVs from Stage 2 of my joint kinematics pipeline, plus a condition-label CSV:

- `ANK_EM2.csv`, `HIP_EM2.csv`, `KNE_EM2.csv` — per-subject ensemble joint angle curves from `joint_kinematics_ensemble_avg.ipynb`
- `Condition.csv` — long-format labels (Subject, Group, Time, Condition) used to align rows with the ensemble curves

## Pipeline context

This notebook sits alongside the descriptive joint kinematics pipeline:

1. `vicon_joint_kinematics_peak_rom.ipynb` — per-trial gait-cycle segmentation
2. `joint_kinematics_ensemble_avg.ipynb` — per-subject ensemble curves
3. `joint_kinematics_group_statistics.ipynb` — peak-value group statistics (descriptive)
4. **This notebook** — time-series SPM tests on the ensemble curves

## What SPM gives me that scalar tests do not

The peak-value statistics in `joint_kinematics_group_statistics.ipynb` ask whether group means differ at the maximum or minimum of the cycle. SPM asks the stronger question: *is there any point in the gait cycle where the group means differ, after correcting for the fact that I am implicitly testing many time points*? This catches differences in shape and timing that peak-value tests miss.

---


## 1. Setup and Data Import

Load the ensemble-mean CSVs for ankle, hip, and knee and the condition-label CSV.

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
import os,sys
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [ ]:
CON = pd.read_csv('Condition.csv')
CON

In [ ]:
ANK = pd.read_csv('ANK_EM2.csv')
ANK = ANK.drop("Unnamed: 0", axis=1)
HIP = pd.read_csv('HIP_EM2.csv')
HIP = HIP.drop("Unnamed: 0", axis=1)
KNE = pd.read_csv('KNE_EM2.csv')
KNE = KNE.drop("Unnamed: 0", axis=1)

## 2. Prepare Data for SPM

Transpose each joint dataframe so that rows are subjects and columns are time points along the gait cycle (this is the orientation SPM expects). Then attach the condition labels (Subject, Group, Time, Condition) as the leading columns.

In [ ]:
ANK_transposed = ANK.T 
HIP_transposed = HIP.T  
KNE_transposed = KNE.T  

In [ ]:
ANK_idx = ANK_transposed.reset_index(drop=True)
HIP_idx = HIP_transposed.reset_index(drop=True)
KNE_idx = KNE_transposed.reset_index(drop=True)

In [ ]:
ANK_df = pd.concat([CON,ANK_idx], axis=1)
HIP_df = pd.concat([CON,HIP_idx], axis=1)
KNE_df = pd.concat([CON,KNE_idx], axis=1)

Save the prepared dataframes for downstream use.

In [ ]:
ANK_df.to_csv('ANK_EM.csv')
HIP_df.to_csv('HIP_EM.csv')
KNE_df.to_csv('KNE_EM.csv')

## 3. SPM Workflow Reference

This section documents the standard SPM1D test types I worked with during the dissertation:

- `spm1d.stats.ttest2(group1, group2)` — independent samples t-test (between-subjects)
- `spm1d.stats.ttest_paired(group1, group2)` — paired t-test (within-subjects)
- `spm1d.stats.anova1(Y, groups)` — one-way ANOVA
- `spm1d.stats.anova2rm(Y, A, B, subj)` — two-way ANOVA with repeated measures

For each test, calling `spm.inference(alpha=0.05)` produces the inference result, which can be plotted with `spm_results.plot()` to show the SPM{t} (or SPM{F}) curve with significance threshold.

In [ ]:
import numpy as np
import pandas as pd
import spm1d
import matplotlib.pyplot as plt

# Sample joint kinematics data (e.g., knee flexion angle)
# Simulating two groups (control vs. experimental)
np.random.seed(42)
n_timepoints = 101  # Number of time points in gait cycle
n_subjects = 15     # Number of subjects per group

# Generate synthetic joint angle data (random variations)
group1 = np.sin(np.linspace(0, 2*np.pi, n_timepoints)) * 30 + np.random.normal(0, 2, (n_subjects, n_timepoints))
group2 = np.sin(np.linspace(0, 2*np.pi, n_timepoints)) * 30 + 5 + np.random.normal(0, 2, (n_subjects, n_timepoints))  # Shifted by 5 degrees

# Convert to NumPy arrays for SPM analysis
Y = np.vstack([group1, group2])  # Combined dataset
groups = np.array([0]*n_subjects + [1]*n_subjects)  # Labels (0 = control, 1 = experimental)

# Perform SPM t-test (two-sample)
spm = spm1d.stats.ttest2(group1, group2)
spm_results = spm.inference(alpha=0.05)

# Plot results
plt.figure(figsize=(10, 5))
spm_results.plot()
plt.title("SPM Analysis of Joint Kinematics (Knee Angle)")
plt.xlabel("Time (%)")
plt.ylabel("t-value")
plt.grid()
plt.show()

## 4. Extract Arrays for Statistical Comparison

Pull the curve values (`X`) and the condition labels (`Group`, `Condition`, `Subject`) out of the ensemble dataframe as numpy arrays — `spm1d` operates on numpy arrays rather than dataframes.

In [ ]:
X = ANK_df.iloc[:,4:].to_numpy()
X

In [ ]:
Group = ANK_df.iloc[:,1].to_numpy()
Group

In [ ]:
Condition = ANK_df.iloc[:,2].to_numpy()
Condition

In [ ]:
Subject = ANK_df.iloc[:,0].to_numpy()
Subject

Verify the group structure before running the test.

In [ ]:
ANK_df.groupby(['Group'])

In [ ]:
group_mean = ANK_df.groupby(['Group']).mean()
group_mean

## 5. Split Curves by Group for Comparison

For an independent-samples comparison, separate the curves into the two groups (Training/Experimental and Control).

In [ ]:
Exp = ANK_df[ANK_df['Group']=='Training'].iloc[:,4:].to_numpy()

Exp

In [ ]:
Con = ANK_df[ANK_df['Group']=='Control'].iloc[:,4:].to_numpy()

Con

Confirm sample sizes per Group × Condition cell.

In [ ]:
df = pd.DataFrame({"Group": Group, "Condition": Condition, "Subject": Subject})
print(df.groupby(["Group", "Condition"])["Subject"].count())


## 6. SPM Independent T-Test (Ankle, Pooled Across Conditions)

Run an independent-samples SPM t-test comparing the Control and Experimental groups on the ankle ensemble curves. The plotted result shows the SPM{t} statistic as a function of percent gait cycle, with the critical threshold marked — any region of the curve exceeding the threshold is a region of statistically significant group difference, after multiple-comparison correction.

In [ ]:
# Perform SPM independent t-test
spm = spm1d.stats.ttest2(Con, Exp)
spm_results = spm.inference(alpha=0.05)

# Plot results
plt.figure(figsize=(10, 5))
spm_results.plot()
plt.title("SPM Independent t-Test: Ankle Joint Angles (Control vs. Experimental)")
plt.xlabel("Time (%)")
plt.ylabel("t-value")
plt.grid()
plt.show()

## 7. Load Per-Joint Ensemble CSVs for Group × Time Analysis

For the Group × Time analysis, reload each joint's ensemble-mean dataframe with the condition labels intact.

In [ ]:
HIP = pd.read_csv('HIP_EM.csv')
HIP = HIP.drop("Unnamed: 0", axis=1)

In [ ]:
KNE = pd.read_csv('KNE_EM.csv')
KNE = KNE.drop("Unnamed: 0", axis=1)

In [ ]:
ANK = pd.read_csv('ANK_EM.csv')
ANK = ANK.drop("Unnamed: 0", axis=1)

## 8. Extract Group × Time Ensemble Curves

For each joint, compute the mean and SD ensemble curves separated by Group (Training/Control) and Time (1/2). These are the four conditions whose curves I will compare.

In [ ]:
aExp1_EM = ANK.loc[(ANK['Group'] == "Training") & (ANK['Time'] == 1), :].iloc[:,4:].mean()
aExp2_EM = ANK.loc[(ANK['Group'] == "Training") & (ANK['Time'] == 2), :].iloc[:,4:].mean()
aCon1_EM = ANK.loc[(ANK['Group'] == "Control") & (ANK['Time'] == 1), :].iloc[:,4:].mean()
aCon2_EM = ANK.loc[(ANK['Group'] == "Control") & (ANK['Time'] == 2), :].iloc[:,4:].mean()

In [ ]:
aExp1_SD = ANK.loc[(ANK['Group'] == "Training") & (ANK['Time'] == 1), :].iloc[:,4:].std()
aExp2_SD = ANK.loc[(ANK['Group'] == "Training") & (ANK['Time'] == 2), :].iloc[:,4:].std()
aCon1_SD = ANK.loc[(ANK['Group'] == "Control") & (ANK['Time'] == 1), :].iloc[:,4:].std()
aCon2_SD = ANK.loc[(ANK['Group'] == "Control") & (ANK['Time'] == 2), :].iloc[:,4:].std()

In [ ]:
kExp1_EM = KNE.loc[(KNE['Group'] == "Training") & (KNE['Time'] == 1), :].iloc[:,4:].mean()
kExp2_EM = KNE.loc[(KNE['Group'] == "Training") & (KNE['Time'] == 2), :].iloc[:,4:].mean()
kCon1_EM = KNE.loc[(KNE['Group'] == "Control") & (KNE['Time'] == 1), :].iloc[:,4:].mean()
kCon2_EM = KNE.loc[(KNE['Group'] == "Control") & (KNE['Time'] == 2), :].iloc[:,4:].mean()

In [ ]:
kExp1_SD = KNE.loc[(KNE['Group'] == "Training") & (KNE['Time'] == 1), :].iloc[:,4:].std()
kExp2_SD = KNE.loc[(KNE['Group'] == "Training") & (KNE['Time'] == 2), :].iloc[:,4:].std()
kCon1_SD = KNE.loc[(KNE['Group'] == "Control") & (KNE['Time'] == 1), :].iloc[:,4:].std()
kCon2_SD = KNE.loc[(KNE['Group'] == "Control") & (KNE['Time'] == 2), :].iloc[:,4:].std()

In [ ]:
hExp1_EM = HIP.loc[(HIP['Group'] == "Training") & (HIP['Time'] == 1), :].iloc[:,4:].mean()
hExp2_EM = HIP.loc[(HIP['Group'] == "Training") & (HIP['Time'] == 2), :].iloc[:,4:].mean()
hCon1_EM = HIP.loc[(HIP['Group'] == "Control") & (HIP['Time'] == 1), :].iloc[:,4:].mean()
hCon2_EM = HIP.loc[(HIP['Group'] == "Control") & (HIP['Time'] == 2), :].iloc[:,4:].mean()

In [ ]:
hExp1_SD = HIP.loc[(HIP['Group'] == "Training") & (HIP['Time'] == 1), :].iloc[:,4:].std()
hExp2_SD = HIP.loc[(HIP['Group'] == "Training") & (HIP['Time'] == 2), :].iloc[:,4:].std()
hCon1_SD = HIP.loc[(HIP['Group'] == "Control") & (HIP['Time'] == 1), :].iloc[:,4:].std()
hCon2_SD = HIP.loc[(HIP['Group'] == "Control") & (HIP['Time'] == 2), :].iloc[:,4:].std()

## 9. Visualize Group × Time Ensemble Curves

Plot mean curves with ±1 SD bands, one figure per joint showing Training Time 2 vs Control Time 2 (or the relevant comparison). The shaded bands give a visual sense of cycle-to-cycle (and subject-to-subject) variability around the mean curves.

In [ ]:
# Plot Two Datasets in One Graph
plt.figure(figsize=(8, 5))
plt.rcParams.update({'font.size': 12}) 

# Plot First Dataset (e.g., Knee Joint)
plt.plot(hExp1_EM.index, hExp1_EM, label="Training T1 Mean", color="g", linewidth =4, linestyle='dotted')
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hExp1_EM.index, hExp1_EM - hExp1_SD , hExp1_EM + hExp1_SD, color="g", alpha=0.2, label="Training T1 SD")

# Plot First Dataset (e.g., Knee Joint)
plt.plot(hExp2_EM.index, hExp2_EM, label="Training T2 Mean", color="g", linewidth =4)
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hExp2_EM.index, hExp2_EM - hExp2_SD , hExp2_EM + hExp2_SD, color="g", alpha=0.2, label="Training T2 SD")

# Plot Second Dataset (e.g., Hip Joint)
plt.plot(hCon1_EM.index, hCon1_EM, label="Control T1 Mean", color="r", linewidth =4, linestyle='dotted')
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hCon1_EM.index, hCon1_EM - hCon1_SD, hCon1_EM + hCon1_SD, color="r", alpha=0.2, label="Control T1 SD")

# Plot Second Dataset (e.g., Hip Joint)
plt.plot(hCon2_EM.index, hCon2_EM, label="Control T2 Mean", color="r", linewidth =4)
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hCon2_EM.index, hCon2_EM - hCon2_SD, hCon2_EM + hCon2_SD, color="r", alpha=0.2, label="Control T1 SD")

# Labels and Legends
plt.xlabel("Gait Cycle (%)")
plt.ylabel("Hip Angle (degrees)")
#plt.title("Ensemble Averaging of Hip Kinematics (Experimental 2nd visit vs Control 1st visit)")
#plt.legend()
plt.show()

In [ ]:
# Plot Two Datasets in One Graph
plt.figure(figsize=(8, 5))
plt.rcParams.update({'font.size': 12}) 

# Plot First Dataset (e.g., Knee Joint)
plt.plot(kExp1_EM.index, kExp1_EM, label="Training T1 Mean", color="g", linewidth =4, linestyle='dotted')
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hExp1_EM.index, hExp1_EM - hExp1_SD , hExp1_EM + hExp1_SD, color="g", alpha=0.2, label="Training T1 SD")

# Plot First Dataset (e.g., Knee Joint)
plt.plot(kExp2_EM.index, kExp2_EM, label="Training T2 Mean", color="g", linewidth =4)
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hExp2_EM.index, hExp2_EM - hExp2_SD , hExp2_EM + hExp2_SD, color="g", alpha=0.2, label="Training T2 SD")

# Plot Second Dataset (e.g., Hip Joint)
plt.plot(kCon1_EM.index, kCon1_EM, label="Control T1 Mean", color="r", linewidth =4, linestyle='dotted')
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hCon1_EM.index, hCon1_EM - hCon1_SD, hCon1_EM + hCon1_SD, color="r", alpha=0.2, label="Control T1 SD")

# Plot Second Dataset (e.g., Hip Joint)
plt.plot(kCon2_EM.index, kCon2_EM, label="Control T2 Mean", color="r", linewidth =4)
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hCon2_EM.index, hCon2_EM - hCon2_SD, hCon2_EM + hCon2_SD, color="r", alpha=0.2, label="Control T1 SD")

# Labels and Legends
plt.xlabel("Gait Cycle (%)")
plt.ylabel("Knee Angle (degrees)")
#plt.title("Ensemble Averaging of Hip Kinematics (Experimental 2nd visit vs Control 1st visit)")
#plt.legend()
plt.show()

In [ ]:
# Plot Two Datasets in One Graph
plt.figure(figsize=(8, 5))
plt.rcParams.update({'font.size': 12}) 

# Plot First Dataset (e.g., Knee Joint)
plt.plot(aExp1_EM.index, aExp1_EM, label="Training T1 Mean", color="g", linewidth =4, linestyle='dotted')
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hExp1_EM.index, hExp1_EM - hExp1_SD , hExp1_EM + hExp1_SD, color="g", alpha=0.2, label="Training T1 SD")

# Plot First Dataset (e.g., Knee Joint)
plt.plot(aExp2_EM.index, aExp2_EM, label="Training T2 Mean", color="g", linewidth =4)
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hExp2_EM.index, hExp2_EM - hExp2_SD , hExp2_EM + hExp2_SD, color="g", alpha=0.2, label="Training T2 SD")

# Plot Second Dataset (e.g., Hip Joint)
plt.plot(aCon1_EM.index, aCon1_EM, label="Control T1 Mean", color="r", linewidth =4, linestyle='dotted')
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hCon1_EM.index, hCon1_EM - hCon1_SD, hCon1_EM + hCon1_SD, color="r", alpha=0.2, label="Control T1 SD")

# Plot Second Dataset (e.g., Hip Joint)
plt.plot(aCon2_EM.index, aCon2_EM, label="Control T2 Mean", color="r", linewidth =4)
plt.xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
#plt.fill_between(hCon2_EM.index, hCon2_EM - hCon2_SD, hCon2_EM + hCon2_SD, color="r", alpha=0.2, label="Control T1 SD")

# Labels and Legends
plt.xlabel("Gait Cycle (%)")
plt.ylabel("Ankle Angle (degrees)")
#plt.title("Ensemble Averaging of Hip Kinematics (Experimental 2nd visit vs Control 1st visit)")
plt.legend()
plt.show()

## 10. SPM Tests on Group × Time Subsets

Run targeted SPM tests for specific comparisons:

- **Paired t-test (Control, Time 1 vs Time 2)** — does the Control group's ankle curve change over time?
- **Independent t-test (Control vs Experimental at Time 2)** — at Time 2, does the ankle curve differ between groups?

Each test produces an SPM curve with the critical threshold marked. Significant regions identify the percentages of the gait cycle where the comparison reaches significance after multiple-comparison correction.

In [ ]:
# Perform SPM dependent t-test
spm = spm1d.stats.ttest_paired(aCon1, aCon2)
spm_results = spm.inference(alpha=0.05)

# Plot results
plt.figure(figsize=(10, 5))
spm_results.plot()
plt.title("SPM Dependent t-Test: Hip Joint Angles Control Group (1st visit vs. 2nd visit)")
plt.xlabel("Time (%)")
plt.ylabel("t-value")
plt.grid()
plt.show()

In [ ]:
# Perform SPM independent t-test
spm = spm1d.stats.ttest2(aCon2, aExp2)
spm_results = spm.inference(alpha=0.05)

# Plot results
plt.figure(figsize=(10, 5))
spm_results.plot()
plt.title("SPM Independent t-Test: Ankle Joint Angles (Control 1st visit vs. Experimental 2nd visit)")
plt.xlabel("Time (%)")
plt.ylabel("t-value")
plt.grid()
plt.show()